In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/VQA Project/images-20260503T095844Z-3-001.zip'
extract_path = '/content/drive/MyDrive/VQA Project/images'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Images extracted")

Images extracted


In [ ]:
img_dir = '/content/drive/MyDrive/VQA Project/images/images/'

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/VQA Project/vqa_dataset.csv')
print(df.head())

              image                       question  \
0  images/img_0.jpg          What is in the image?   
1  images/img_0.jpg          What object is shown?   
2  images/img_0.jpg  What does the image describe?   
3  images/img_1.jpg          What is in the image?   
4  images/img_1.jpg          What object is shown?   

                                answer  
0                                  dog  
1                                  dog  
2  dog running on beach during daytime  
3                                  dog  
4                                  dog  


In [ ]:
df['image'] = df['image'].apply(lambda x: img_dir + x.split('/')[-1])
print(df['image'].head())

0    /content/drive/MyDrive/VQA Project/images/imag...
1    /content/drive/MyDrive/VQA Project/images/imag...
2    /content/drive/MyDrive/VQA Project/images/imag...
3    /content/drive/MyDrive/VQA Project/images/imag...
4    /content/drive/MyDrive/VQA Project/images/imag...
Name: image, dtype: object


In [ ]:
import os

print(df['image'][0])
print(os.path.exists(df['image'][0]))

/content/drive/MyDrive/VQA Project/images/images/img_0.jpg
True


# **Load** **ResNet**

In [ ]:
import torch
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained ResNet
resnet = models.resnet50(pretrained=True)

# Remove classification layer
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])

resnet = resnet.to(device)
resnet.eval()

print("✅ ResNet loaded!")
print("Output shape per image: 2048")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 253MB/s]

✅ ResNet loaded!
Output shape per image: 2048


In [ ]:
from torchvision import transforms
from PIL import Image
import numpy as np
import os

IMG_SIZE = (224, 224)

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor()
])

# **Feature** **extraction**

In [ ]:
def extract_features(img_path):
    img = Image.open(img_path).convert('RGB')
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        features = resnet(img)

    return features.squeeze().cpu().numpy()  # (2048,)

In [ ]:
features_cache = {}
all_features = []

for idx, row in df.iterrows():
    img_name = os.path.basename(row['image'])
    full_path = os.path.join('/content/drive/MyDrive/VQA Project/images/images/', img_name)

    if img_name not in features_cache:
        features_cache[img_name] = extract_features(full_path)

    all_features.append(features_cache[img_name])

    if (idx + 1) % 100 == 0:
        print(f"{idx+1}/{len(df)} done...")

100/585 done...
200/585 done...
300/585 done...
400/585 done...
500/585 done...


In [ ]:
import numpy as np

all_features = np.array(all_features)

print("✅ Features extracted!")
print("Shape:", all_features.shape)

✅ Features extracted!
Shape: (585, 2048)


In [ ]:
print("\n Sample feature vector (first 10 values):")
print(all_features[0][:10])
print(f"\nMin: {all_features.min():.4f} | Max: {all_features.max():.4f}")


 Sample feature vector (first 10 values):
[0.21760142 0.21996513 0.53800297 0.62351006 0.3082157  1.0412934
 1.419896   1.3345016  0.20205906 0.40792054]

Min: 0.0000 | Max: 6.3737


In [ ]:
import numpy as np

SAVE_PATH = "/content/drive/MyDrive/resnet_features.npy"

np.save(SAVE_PATH, all_features)

print("Features saved to:", SAVE_PATH)

Features saved to: /content/drive/MyDrive/resnet_features.npy
